# Pandas Workout: Practice Until It Is Automatic
**Companion to:** *Pandas from Dictionary to Clean CSV: A Complete Course*  |  **Tasks:** 32 + 1 challenge  |  **Time:** 2 to 4 hours

## How this workout works
1. You get a **new dataset** (company staff records) so you practise *thinking*, not remembering the lesson's example
2. Each task says **what** to do, never **how**. Write your own code in the empty cell below the task
3. Use the **exact variable names** in **bold**: the self-checks depend on them
4. After each task, run the grey **Self-check** cell: ✅ means you match the expected result, ❌ means try again. Checks never reveal the answer
5. Tasks build on each other, so work **top to bottom**
6. Stuck? Read the 💡 hint, re-read the matching lesson module, then try again. Solutions are at the very **bottom**. Look only after a real attempt

## Difficulty tags
**Beginner:** one step, directly from the lesson · **Intermediate:** combines two ideas · **Advanced:** needs you to decide the method

## The workout map

| Part | Skills practised | Tasks |
|---|---|---|
| 1 | Create and inspect | 1 to 3 |
| 2 | Types and text | 4 to 6 |
| 3 | Duplicates | 7 to 8 |
| 4 | Missing values | 9 to 13 |
| 5 | Outliers and invalid values | 14 to 18 |
| 6 | Feature engineering | 19 to 24 |
| 7 | Summaries | 25 to 27 |
| 8 | Charts | 28 to 30 |
| 9 | Save and verify | 31 to 32 |
| Challenge | Build a reusable cleaning function | 33 |

## Setup
Run the next two cells. Do not change them.

In [ ]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


def _norm(v):
    if isinstance(v, dict):
        return {str(k): _norm(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_norm(x) for x in v]
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return round(float(v), 2)
    return str(v)


def check(task, get_value, expected):
    """Self-check helper: prints whether your result matches the expected answer (without revealing it)."""
    try:
        got = hashlib.sha256(json.dumps(_norm(get_value()), sort_keys=True).encode()).hexdigest()[:8]
    except Exception as e:
        print(f"❌ {task}: could not run the check ({type(e).__name__}). "
              "Did you finish the task and use the required variable names?")
        return
    print(f"✅ {task}: looks correct" if got == expected
          else f"❌ {task}: your result does not match. Re-read the task and try again.")


def frames_match(a, b):
    """True if two DataFrames hold the same values (used in the final challenge)."""
    try:
        pd.testing.assert_frame_equal(a, b, check_dtype=False)
        return True
    except AssertionError:
        return False

### The data
One dictionary holding 43 staff records. Read it first and try to spot problems.

In [ ]:
# Staff records for a mid-sized company: a DICTIONARY OF LISTS. Run this cell; do NOT edit it.
# It is deliberately messy: typos, missing values, duplicates and impossible numbers.
workout_dict = {
    "staff_id": [
        'S101', 'S102', 'S103', 'S104', 'S105',
        'S106', 'S107', 'S108', 'S109', 'S110',
        'S111', 'S112', 'S113', 'S114', 'S115',
        'S116', 'S117', 'S118', 'S119', 'S120',
        'S121', 'S122', 'S123', 'S124', 'S125',
        'S126', 'S127', 'S128', 'S129', 'S130',
        'S131', 'S132', 'S133', 'S134', 'S135',
        'S136', 'S137', 'S138', 'S139', 'S140',
        'S106', 'S115', 'S123',
    ],
    "full_name": [
        'Kojo Boateng', 'Kofi Owusu', 'Abena Annan', 'Efua Boateng', 'Efua Osei',
        'Kwesi Osei', 'Esi Osei', 'Akosua Annan', 'Kofi Frimpong', 'Kwame Serwaa',
        'Kwame Serwaa', 'Kwame Quartey', 'Efua Quartey', 'Abena Owusu', 'Kojo Darko',
        'Esi Darko', 'Efua Owusu', 'Esi Frimpong', 'Yaw Frimpong', 'Kojo Owusu',
        'Nana Frimpong', 'Efua Annan', 'Nana Mensah', 'Kwame Annan', 'Akosua Frimpong',
        'Kojo Osei', 'Kojo Boateng', 'Esi Darko', 'Kojo Osei', 'Kofi Serwaa',
        'Nana Frimpong', 'Abena Darko', 'Kojo Quartey', 'Nana Boateng', 'Akosua Owusu',
        'Kofi Owusu', 'Kofi Serwaa', 'Kojo Darko', 'Kofi Mensah', 'Esi Frimpong',
        'Kwesi Osei', 'Kojo Darko', 'Nana Mensah',
    ],
    "department": [
        'Information Technology', 'Finance', 'Sales', '  FINANCE', 'Finance',
        'Human Resources', 'Information Technology', 'Sales', 'Finance', 'Operations',
        'Sales', 'Information Technology', 'Operations', 'Human Resources', 'Sales',
        'Human Resources', 'Information Technology', 'Finance', 'Information Technology', 'Human Resources',
        'Operations', 'Sales', 'Human Resources', 'sales ', np.nan,
        'Human Resources', 'Information Technology', np.nan, 'Human Resources', 'Finance',
        'Information Technology', 'Information Technology', 'Human Resources', 'operations', 'Operations',
        'Finance', 'Information Technology', 'Finance', 'Sales', 'Information Technology',
        'Human Resources', 'Sales', 'human resources ',
    ],
    "job_title": [
        'Data Analyst', 'Accountant', 'Account Manager', 'Developer', 'Financial Analyst',
        'Recruiter', 'Developer', 'Sales Executive', 'Financial Analyst', 'Logistics Coordinator',
        'Account Manager', 'Support Engineer', 'Logistics Coordinator', 'HR Officer', 'Sales Executive',
        'HR Officer', 'Data Analyst', 'Accountant', 'Developer', 'HR Officer',
        'Operations Officer', 'Sales Executive', 'Recruiter', 'Data Analyst', 'Account Manager',
        'HR Officer', 'Support Engineer', 'Sales Executive', 'Recruiter', 'Accountant',
        'Support Engineer', 'Data Analyst', 'HR Officer', 'Accountant', 'Logistics Coordinator',
        'Financial Analyst', 'Support Engineer', 'Accountant', 'Sales Executive', 'Data Analyst',
        'Recruiter', 'Sales Executive', 'Recruiter',
    ],
    "city": [
        'Accra', 'Tamale', 'Takoradi', 'Tamale', 'accra',
        'Kumasi', 'tamale', 'Accra', 'Accra', 'Tamale',
        'Tamale', 'Kumasi', 'Tamale', 'Takoradi', 'Kumasi',
        'Takoradi', 'Kumasi', 'Tamale', 'Accra', np.nan,
        'Takoradi', 'Accra', 'Takoradi', 'Kumasi', 'Accra',
        np.nan, 'Takoradi', 'Kumasi', 'KUMASI ', 'Accra',
        'Tamale', 'Accra', np.nan, 'Tamale', 'Tamale',
        'Accra', 'Accra', 'Accra', 'Takoradi', 'Takoradi',
        'Kumasi', 'Kumasi', ' TAKORADI',
    ],
    "salary": [
        6032.65, 5540.82, 5602.78, 8108.78, 6602.97,
        4081.82, 7775.22, 5280.65, 5737.58, 4342.13,
        5824.32, 4800.08, 3922.61, 4328.76, 4488.76,
        np.nan, 6474.25, np.nan, 8369.07, 4388.37,
        4073.19, 4985.09, 4583.2, 6358.02, 5959.04,
        4763.23, 4313.04, 4484.74, 3940.71, 5846.03,
        3971.59, 273062.8, 4352.17, 5296.97, 4511.25,
        6797.62, 4514.9, 206465.6, np.nan, 6248.29,
        4081.82, 4488.76, 4583.2,
    ],
    "bonus": [
        301.63, np.nan, 280.14, 0.0, 990.45,
        0.0, 1166.28, np.nan, 860.64, 217.11,
        np.nan, 240.0, 392.26, np.nan, 673.31,
        190.73, 647.43, 890.98, 0.0, np.nan,
        610.98, 249.25, 229.16, 953.7, 595.9,
        238.16, 646.96, 224.24, 591.11, 584.6,
        397.16, 682.66, 435.22, 794.55, 451.12,
        1019.64, 0.0, 516.16, 0.0, np.nan,
        0.0, 673.31, 229.16,
    ],
    "hire_date": [
        '2016-04-05', '2017-04-19', '2017-05-11', '2015-12-19', '2015-06-19',
        '2023-11-10', '2018-03-12', '2018-01-01', '2021-11-10', '2023-12-10',
        '2020-01-01', '2024-03-15', '2023-01-15', '2021-07-02', '2024-12-25',
        '2019-11-17', '2018-11-16', '2022-04-25', '2018-02-09', '2024-06-29',
        '2020-10-28', '2022-12-01', '2023-03-07', '2021-02-14', '2017-08-21',
        '2019-09-09', '2017-09-07', '2022-03-25', '2024-03-15', '2017-12-25',
        '2016-08-09', '2016-05-07', '2024-08-06', '2017-01-09', '2015-02-23',
        '2021-08-20', '2024-02-06', '2019-06-27', '2021-01-31', '2019-10-17',
        '2023-11-10', '2024-12-25', '2023-03-07',
    ],
    "age": [
        57.0, 28.0, 36.0, 56.0, 39.0,
        42.0, 42.0, 29.0, 55.0, 54.0,
        34.0, 42.0, 52.0, 33.0, 42.0,
        46.0, np.nan, 40.0, 47.0, 57.0,
        39.0, 5.0, 54.0, 32.0, 33.0,
        np.nan, 58.0, 47.0, 49.0, 51.0,
        50.0, 37.0, 39.0, 34.0, np.nan,
        250.0, 26.0, 29.0, 32.0, 33.0,
        42.0, 42.0, 54.0,
    ],
    "performance_score": [
        2.0, 1.0, 4.0, 4.0, 3.0,
        2.0, 2.0, 1.0, 3.0, 1.0,
        1.0, np.nan, 3.0, 1.0, 5.0,
        2.0, np.nan, 2.0, 3.0, 3.0,
        np.nan, 1.0, 4.0, 1.0, 1.0,
        1.0, np.nan, 4.0, 3.0, 4.0,
        4.0, 5.0, np.nan, 4.0, 3.0,
        3.0, 3.0, 1.0, 4.0, 3.0,
        2.0, 5.0, 4.0,
    ],
    "remote": [
        'Yes', 'Yes', 'Yes', 'Yes', 'No',
        'No', 'Yes', 'Yes', 'No', 'No',
        'Yes', 'No', 'Yes', 'Yes', 'Yes',
        'No', 'Yes', 'No', 'No', 'Yes',
        'No', 'No', 'Yes', 'Yes', 'No',
        'No', 'Yes', 'Yes', 'Yes', 'Yes',
        'Yes', 'Yes', 'Yes', 'Yes', 'Yes',
        'Yes', 'Yes', 'Yes', 'No', 'Yes',
        'No', 'Yes', 'Yes',
    ],
}

---

---
## Part 1: Create and inspect
**Skills:** DataFrame from a dictionary, first look at the data.

### Task 1: Create the DataFrame  `Beginner`

Convert `workout_dict` into a DataFrame named **`staff`**.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 1", lambda: staff.shape, "72fa6c52")

### Task 2: View the first 8 records  `Beginner`

Display the **first 8 rows** of `staff`.

In [ ]:
# Write your code below

### Task 3: Summarise the structure  `Beginner`

Display each column's data type and non-null count.

**Write below:** which columns have the wrong type for their meaning, and which contain missing values?

In [ ]:
# Write your code below

**Your answer:** _(double-click to write here)_

---
## Part 2: Types and text
**Skills:** standardising text, converting dates and yes/no values.

### Task 4: Standardise text  `Beginner`

In the columns **`department`** and **`city`**: remove spaces at the start and end, collapse repeated inner spaces, and apply Title Case. Missing values must stay missing. Update `staff`.

💡 *Hint: Chain the `.str` methods you learned in Module 5.*

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 4", lambda: (sorted(staff["department"].dropna().unique()), sorted(staff["city"].dropna().unique())), "d1fca658")

### Task 5: Convert the hire date  `Beginner`

Convert **`hire_date`** to a real **datetime** column.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 5", lambda: pd.api.types.is_datetime64_any_dtype(staff["hire_date"]), "b5bea41b")

### Task 6: Yes/No to True/False  `Intermediate`

Create a new True/False column **`is_remote`** that is `True` where `remote` is "Yes".

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 6", lambda: int(staff["is_remote"].sum()), "35135aaa")

---
## Part 3: Duplicates
**Skills:** counting and removing duplicates, including hidden ones.

### Task 7: Count duplicates  `Beginner`

Count the **fully duplicated rows** now (after text standardisation). Store the number in **`dup_count`**.

**Write below:** why is this number larger than it would have been before Task 4?

In [ ]:
# Write your code below

**Your answer:** _(double-click to write here)_

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 7", lambda: int(dup_count), "4e074085")

### Task 8: Remove duplicates  `Beginner`

Drop the duplicated rows (keep the first) and **reset the index**. Update `staff`.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 8", lambda: staff.shape, "c2a4a6f9")
check("Task 8", lambda: bool(staff["staff_id"].is_unique), "b5bea41b")

---
## Part 4: Missing values
**Skills:** auditing gaps and choosing a sensible treatment for each column.

### Task 9: Count missing values  `Beginner`

Count the missing values in **each column** and store the result in **`missing_counts`**.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 9", lambda: missing_counts.sort_index().to_dict(), "d1b70370")

### Task 10: Fill department and city  `Beginner`

Fill missing **`department`** and **`city`** values with the text **"Unknown"**. Update `staff`.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 10", lambda: (int((staff["department"] == "Unknown").sum()), int((staff["city"] == "Unknown").sum())), "5bf77553")

### Task 11: Fill bonus  `Beginner`

Assume a missing **`bonus`** means **no bonus was paid** (0). Update `staff`.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 11", lambda: (int(staff["bonus"].isna().sum()), staff["bonus"].sum()), "0c85f907")

### Task 12: Fill age by department  `Intermediate`

Fill missing **`age`** with the **median age of the person's own department**. If any gaps remain (a department with no known ages), fill them with the overall median age. Update `staff`.

💡 *Hint: Group-aware filling: `groupby(...).transform(...)` returns a value for every row.*

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 12", lambda: (int(staff["age"].isna().sum()), staff["age"].sum()), "ee5dd651")

### Task 13: Keep missing scores, but flag them  `Intermediate`

A missing **`performance_score`** may mean the review has not happened yet, so **keep it missing**. Instead, create a True/False column **`performance_missing`** marking those rows.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 13", lambda: (int(staff["performance_missing"].sum()), int(staff["performance_score"].isna().sum())), "1d177438")

---
## Part 5: Outliers and invalid values
**Skills:** business rules, group-aware outlier detection and treatment.

### Task 14: Fill missing salaries  `Intermediate`

Fill missing **`salary`** with the **median salary of the person's department**. Use the overall median if any gaps remain. Update `staff`.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 14", lambda: (int(staff["salary"].isna().sum()), staff["salary"].sum()), "8f8d5db2")

### Task 15: Find impossible ages  `Beginner`

Staff must be between **18 and 70** years old. Count the rows that break this rule and store the number in **`invalid_age_count`**.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 15", lambda: int(invalid_age_count), "d4735e3a")

### Task 16: Repair impossible ages  `Intermediate`

Set the invalid ages to missing, then fill them with the **median age of their department** (overall median if needed). Update `staff`.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 16", lambda: (bool(staff["age"].between(18, 70).all()), staff["age"].sum()), "01afb200")

### Task 17: Detect salary outliers within departments  `Advanced`

Compare each salary with the **median salary of its own department**. Flag a salary as an outlier if it is **more than 3 times** that median (or less than a third). Store the number of flagged rows in **`n_salary_outliers`**.

💡 *Hint: Recall the ratio-to-median rule from Module 8. Look at the flagged rows before deciding they are errors.*

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 17", lambda: int(n_salary_outliers), "d4735e3a")

### Task 18: Treat the salary outliers  `Advanced`

These salaries look like data-entry errors (extra digits). Replace the flagged salaries with their **department median**. Update `staff`, then run the cell below to save a snapshot of your cleaned table.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 18", lambda: (staff["salary"].max(), staff["salary"].sum()), "ad5fb65a")

**Run this cell** to save a snapshot of your cleaned table (used by the final challenge):

In [ ]:
try:
    staff_snapshot = staff.copy()
    print("Snapshot saved.")
except NameError:
    print("Finish Tasks 1 to 18 first, then run this cell again.")

---
## Part 6: Feature engineering
**Skills:** calculations, bins, conditions, group statistics, encoding.

### Task 19: Tenure in years  `Intermediate`

Create **`tenure_years`**: years between **`hire_date`** and the fixed date **31 December 2025**, using 365.25 days per year, rounded to 1 decimal place.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 19", lambda: staff["tenure_years"].sum(), "18ef810f")

### Task 20: Total pay  `Beginner`

Create **`total_pay`** = `salary` + `bonus`, rounded to 2 decimals.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 20", lambda: staff["total_pay"].sum(), "4d43f907")

### Task 21: Age groups  `Intermediate`

Create **`age_group`** with these bands: **18-29**, **30-39**, **40-49**, **50+** (use exactly these labels).

💡 *Hint: Think about which function creates bins with bounds you choose.*

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 21", lambda: staff["age_group"].value_counts().sort_index().to_dict(), "d5b10c97")

### Task 22: Salary compared with the department average  `Advanced`

Create **`salary_vs_dept_avg`** = each person's `salary` divided by the **average salary of their department**, rounded to 2 decimals.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 22", lambda: staff["salary_vs_dept_avg"].sum(), "1ae725e9")

### Task 23: Seniority levels  `Intermediate`

Create **`seniority`**: **"Junior"** if `tenure_years` is below 2, **"Mid"** if below 6, otherwise **"Senior"**.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 23", lambda: pd.Series(staff["seniority"]).value_counts().sort_index().to_dict(), "b04bfe57")

### Task 24: Prepare a table for machine learning  `Advanced`

Build a new DataFrame **`staff_ml`** from the columns `salary`, `bonus`, `age`, `tenure_years`, `department` and `city`. Convert `department` and `city` to 0/1 columns (one-hot encoding), **dropping the first category of each** and using integer values.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 24", lambda: staff_ml.shape, "d959df6c")

---
## Part 7: Summaries
**Skills:** groupby, pivot tables.

### Task 25: Average pay by department  `Intermediate`

Calculate the **average `total_pay` per department**, rounded to 2 decimals and sorted from highest to lowest. Store it in **`avg_pay_by_dept`**.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 25", lambda: avg_pay_by_dept.sort_index().to_dict(), "8d3775ae")

### Task 26: Headcount table  `Intermediate`

Build a table **`headcount`** with **departments as rows** and **cities as columns**, counting staff (`staff_id`) in each combination. Show 0 instead of missing.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 26", lambda: int(headcount.values.sum()), "d59eced1")
check("Task 26", lambda: headcount.shape, "45a166b5")

### Task 27: Remote work share  `Advanced`

For each department calculate the **percentage of staff who work remotely**, rounded to 1 decimal. Store it in **`remote_share_by_dept`**.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 27", lambda: remote_share_by_dept.sort_index().to_dict(), "9d9c81fc")

---
## Part 8: Charts
**Skills:** choosing the right chart and labelling it clearly. Every chart needs a **title** and **labelled axes**.

### Task 28: Box plot of salary  `Beginner`

Draw a **box plot of `salary` by department**. Rotate the labels if they overlap.

In [ ]:
# Write your code below

### Task 29: Histogram of tenure  `Beginner`

Draw a **histogram of `tenure_years`**.

In [ ]:
# Write your code below

### Task 30: Bar chart of headcount by city  `Intermediate`

Draw a **bar chart of the number of staff in each city**, sorted from highest to lowest. Bars must start at zero.

In [ ]:
# Write your code below

---
## Part 9: Save and verify
**Skills:** exporting to CSV and checking the file.

### Task 31: Save to CSV  `Beginner`

Save `staff` to **`staff_clean.csv`** **without** the index column.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 31", lambda: os.path.exists("staff_clean.csv"), "b5bea41b")

### Task 32: Reload and verify  `Intermediate`

Read `staff_clean.csv` into **`staff_check`**, parsing `hire_date` as a **date**. Display its shape and compare with `staff`.

In [ ]:
# Write your code below

In [ ]:
# Self-check (do not edit): run after completing the task
check("Task 32", lambda: (staff_check.shape, pd.api.types.is_datetime64_any_dtype(staff_check["hire_date"])), "76b30c4f")

---
## Challenge (Task 33): build a reusable cleaning function  `Advanced`
Write a function **`clean_staff(workout_dict)`** that takes the raw dictionary and returns a DataFrame identical to your `staff_snapshot`, by repeating the steps from **Tasks 1 to 18** in the same order (create the DataFrame, standardise text and dates, create `is_remote`, remove duplicates, treat missing values, create `performance_missing`, repair ages and salaries).

💡 *Hint: copy your working code into the function body, replace `staff` with a local variable, and `return` it.*

In [ ]:
# Write your function below

In [ ]:
# Self-check (do not edit)
check("Task 33", lambda: frames_match(clean_staff(workout_dict), staff_snapshot), "b5bea41b")

---
## Your score card
Count the tasks where **every** self-check shows ✅ (there are **28** self-checked tasks, including the challenge). Chart tasks have no check: compare your charts with the task description yourself.

| Tasks passed | Meaning |
|---|---|
| 25 to 28 | **Mastery.** You can clean data independently. Try a real-world dataset next |
| 20 to 24 | **Strong.** Revisit the modules for the parts where you saw ❌ |
| 11 to 19 | **Developing.** Re-run the lesson notebook, changing the code as you go, then retry the workout |
| Under 11 | **Foundations needed.** Work through Modules 1 to 7 of the course again, one module per day |

## Reflection
1. Which part was hardest, and what finally helped?
2. Which decision (not just code) took the most thought: filling, flagging, or treating an outlier?
3. What would you do differently if this were your own company's data?

---
# Solutions

⚠️ **Only open these after you have genuinely tried each task.** Alternative correct approaches are fine as long as the self-check passes.

### Task 1: Create the DataFrame
```python
staff = pd.DataFrame(workout_dict)
```

### Task 2: View the first 8 records
```python
staff.head(8)
```

### Task 3: Summarise the structure
```python
staff.info()
```
**Answer:** `hire_date` is stored as text instead of a date; `age`, `salary`, `bonus` and `performance_score` are floats because of missing values (ages and scores should be whole numbers). Missing values exist in `department`, `city`, `salary`, `bonus`, `age` and `performance_score`.

### Task 4: Standardise text
```python
for col in ["department", "city"]:
    staff[col] = staff[col].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
```

### Task 5: Convert the hire date
```python
staff["hire_date"] = pd.to_datetime(staff["hire_date"])
```

### Task 6: Yes/No to True/False
```python
staff["is_remote"] = staff["remote"] == "Yes"
```

### Task 7: Count duplicates
```python
dup_count = staff.duplicated().sum()
```
**Answer:** one duplicate differed from its original only by capitalisation and spaces (for example "information technology " vs "Information Technology"). It became an exact duplicate only after the text was standardised, which is why text cleaning comes before duplicate removal.

### Task 8: Remove duplicates
```python
staff = staff.drop_duplicates().reset_index(drop=True)
```

### Task 9: Count missing values
```python
missing_counts = staff.isna().sum()
```

### Task 10: Fill department and city
```python
staff["department"] = staff["department"].fillna("Unknown")
staff["city"] = staff["city"].fillna("Unknown")
```

### Task 11: Fill bonus
```python
staff["bonus"] = staff["bonus"].fillna(0)
```

### Task 12: Fill age by department
```python
staff["age"] = staff["age"].fillna(staff.groupby("department")["age"].transform("median"))
staff["age"] = staff["age"].fillna(staff["age"].median())
```

### Task 13: Keep missing scores, but flag them
```python
staff["performance_missing"] = staff["performance_score"].isna()
```

### Task 14: Fill missing salaries
```python
staff["salary"] = staff["salary"].fillna(staff.groupby("department")["salary"].transform("median"))
staff["salary"] = staff["salary"].fillna(staff["salary"].median())
```

### Task 15: Find impossible ages
```python
invalid_age_count = (~staff["age"].between(18, 70)).sum()
```

### Task 16: Repair impossible ages
```python
staff.loc[~staff["age"].between(18, 70), "age"] = np.nan
staff["age"] = staff["age"].fillna(staff.groupby("department")["age"].transform("median"))
staff["age"] = staff["age"].fillna(staff["age"].median())
```

### Task 17: Detect salary outliers within departments
```python
dept_median = staff.groupby("department")["salary"].transform("median")
salary_ratio = staff["salary"] / dept_median
salary_outlier = (salary_ratio > 3) | (salary_ratio < 1 / 3)
n_salary_outliers = salary_outlier.sum()
```

### Task 18: Treat the salary outliers
```python
staff.loc[salary_outlier, "salary"] = dept_median[salary_outlier]
```

### Task 19: Tenure in years
```python
staff["tenure_years"] = ((pd.Timestamp("2025-12-31") - staff["hire_date"]).dt.days / 365.25).round(1)
```

### Task 20: Total pay
```python
staff["total_pay"] = (staff["salary"] + staff["bonus"]).round(2)
```

### Task 21: Age groups
```python
staff["age_group"] = pd.cut(staff["age"], bins=[17, 29, 39, 49, 100], labels=["18-29", "30-39", "40-49", "50+"])
```

### Task 22: Salary compared with the department average
```python
staff["salary_vs_dept_avg"] = (staff["salary"] / staff.groupby("department")["salary"].transform("mean")).round(2)
```

### Task 23: Seniority levels
```python
staff["seniority"] = np.select([staff["tenure_years"] < 2, staff["tenure_years"] < 6], ["Junior", "Mid"], default="Senior")
```

### Task 24: Prepare a table for machine learning
```python
staff_ml = pd.get_dummies(staff[["salary", "bonus", "age", "tenure_years", "department", "city"]],
                          columns=["department", "city"], drop_first=True, dtype=int)
```

### Task 25: Average pay by department
```python
avg_pay_by_dept = staff.groupby("department")["total_pay"].mean().round(2).sort_values(ascending=False)
```

### Task 26: Headcount table
```python
headcount = pd.pivot_table(staff, index="department", columns="city", values="staff_id", aggfunc="count", fill_value=0)
```

### Task 27: Remote work share
```python
remote_share_by_dept = (staff.groupby("department")["is_remote"].mean() * 100).round(1)
```

### Task 28: Box plot of salary
```python
plt.figure(figsize=(9, 4))
sns.boxplot(data=staff, x="department", y="salary", color="#9DB4C0")
plt.title("Salary by department")
plt.xlabel("Department")
plt.ylabel("Monthly salary (GHS)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()
```

### Task 29: Histogram of tenure
```python
plt.figure(figsize=(7, 4))
sns.histplot(staff["tenure_years"], bins=10, color="#1F3A5F")
plt.title("How long staff have worked here")
plt.xlabel("Tenure (years)")
plt.ylabel("Number of staff")
plt.show()
```

### Task 30: Bar chart of headcount by city
```python
staff["city"].value_counts().plot(kind="bar", color="#2E7D6B", figsize=(6, 4))
plt.title("Headcount by city")
plt.xlabel("City")
plt.ylabel("Number of staff")
plt.xticks(rotation=0)
plt.show()
```

### Task 31: Save to CSV
```python
staff.to_csv("staff_clean.csv", index=False)
```

### Task 32: Reload and verify
```python
staff_check = pd.read_csv("staff_clean.csv", parse_dates=["hire_date"])
staff_check.shape
```

### Task 33: Challenge
```python
def clean_staff(workout_dict):
    d = pd.DataFrame(workout_dict)

    # text and types
    for col in ["department", "city"]:
        d[col] = d[col].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
    d["hire_date"] = pd.to_datetime(d["hire_date"])
    d["is_remote"] = d["remote"] == "Yes"

    # duplicates
    d = d.drop_duplicates().reset_index(drop=True)

    # missing values
    d["department"] = d["department"].fillna("Unknown")
    d["city"] = d["city"].fillna("Unknown")
    d["bonus"] = d["bonus"].fillna(0)
    d["age"] = d["age"].fillna(d.groupby("department")["age"].transform("median"))
    d["age"] = d["age"].fillna(d["age"].median())
    d["performance_missing"] = d["performance_score"].isna()

    # salaries and ages
    d["salary"] = d["salary"].fillna(d.groupby("department")["salary"].transform("median"))
    d["salary"] = d["salary"].fillna(d["salary"].median())
    d.loc[~d["age"].between(18, 70), "age"] = np.nan
    d["age"] = d["age"].fillna(d.groupby("department")["age"].transform("median"))
    d["age"] = d["age"].fillna(d["age"].median())
    med = d.groupby("department")["salary"].transform("median")
    ratio = d["salary"] / med
    flag = (ratio > 3) | (ratio < 1 / 3)
    d.loc[flag, "salary"] = med[flag]
    return d
```